In [7]:
import pandas as pd
import scipy.stats
import skdim
import umap
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
import numpy as np
from sklearn.decomposition import PCA
import torch
import os
import sys
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.ndimage import gaussian_filter
from BWM.models.isometric_autoencoder import get_density_proxy
# Add project root to path
sys.path.append(os.path.abspath('../../'))
from matplotlib.ticker import MaxNLocator
from BWM.bwm_functions import *
from BWM.models.isometric_autoencoder import SklearnIsometricAutoencoder

plt.rcParams["axes.grid"] = False
matplotlib.rcParams['axes.linewidth'] = .5

# Set global plotting parameters for font size
tick_fontsize = 6
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': tick_fontsize,
    'ytick.labelsize': tick_fontsize,
    'legend.fontsize': 7,
    'figure.titlesize': 7
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


In [8]:
pseudo_response_tensor = np.load('/Users/cl1704/PycharmProjects/embedding_dimension/BWM/ResponseData/CP_pseudo_responses.npy')


In [9]:
def iso_ae_preprocessing(pseudo_response_tensor):
    response_tensor = np.mean(pseudo_response_tensor, axis=0)
    # Z-score and reshape
    response_tensor_norm = (response_tensor - np.mean(response_tensor, axis=(0, 1), keepdims=True)) / (np.std(response_tensor, axis=(0, 1), keepdims=True) + 1e-8)
    response_matrix = response_tensor_norm.reshape(-1, response_tensor_norm.shape[2])

    X_transposed = response_matrix.T

    # Scale each row (neuron) to unit length
    row_norms = np.linalg.norm(X_transposed, axis=1, keepdims=True)
    X_pre_pca = X_transposed / (row_norms + 1e-8)

    # Reduce features to 20 using PCA

    return X_pre_pca

X_input = iso_ae_preprocessing(pseudo_response_tensor)

In [10]:
def calculate_speed(response_matrix):
    """
    Calculate Mean Standardized Derivative using finite differences.
    responses: (conditions, time, neurons)
    """
    response_matrix_expand = response_matrix.reshape( response_matrix.shape[0],100,2)
    diff_responses = np.diff(response_matrix_expand, axis=1)
    # Absolute standardized derivative (already standardized in the previous cell)
    # Mean over conditions and time and pseudo trials
    msd = np.mean(np.abs(diff_responses), axis=(1, 2))
    return msd

speed = calculate_speed(X_input)

In [ ]:
z_neurons = umap.UMAP(densmap=True,dens_lambda=10,n_neighbors = 500).fit_transform(X_input)


In [ ]:
# Compute intrinsic dimension
X_input_low = z_neurons[speed<.01,:]
fisher = skdim.id.FisherS().fit(X_input_low)
id_fisher = fisher.dimension_
print(f"Intrinsic dimension (FisherS): {id_fisher:.2f}")

# Compute local intrinsic dimension
l_id_fisher = skdim.id.FisherS().fit_pw(X_input_low, n_neighbors=25).dimension_pw_
print(f"Mean local intrinsic dimension: {np.nanmean(l_id_fisher):.2f}")


In [ ]:
# Plot distribution of local intrinsic dimension
plt.figure(figsize=(4, 3))
sns.histplot(l_id_fisher, kde=True, color='skyblue', bins=20)
plt.axvline(np.nanmean(l_id_fisher), color='red', linestyle='--', label=f'Mean: {np.nanmean(l_id_fisher):.2f}')
plt.axvline(id_fisher, color='green', linestyle='-', label=f'Global: {id_fisher:.2f}')
plt.xlabel('Local Intrinsic Dimension')
plt.ylabel('Count')
plt.title('Distribution of Local Intrinsic Dimension')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()
